### 🧠 What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence

## Step 1: Load and embed the document

In [5]:
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
chunks = splitter.split_documents(docs)

In [6]:
embedding = OllamaEmbeddings(model = "nomic-embed-text-v2-moe:latest")

In [7]:
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7})

In [9]:
llm = ChatOllama(model="gemma3:latest")
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, model='gemma3:latest')

## Step 3: Query decomposition

In [10]:
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [11]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question=decomposition_chain.invoke({"question": query})


In [12]:
print(decomposition_question)

Okay, here’s a breakdown of the complex question “How does LangChain use memory and agents compared to CrewAI?” into smaller, more manageable sub-questions, designed to improve document retrieval:

1.  **What are the memory capabilities of LangChain?** (This focuses specifically on LangChain's implementation of memory – types of memory, how it's integrated, and its purpose.)

2.  **How do LangChain agents work and what are their key components?** (This drills down into LangChain’s agent framework: tool selection, reasoning steps, and overall agent architecture.)

3.  **What is CrewAI's approach to memory and agents?** (This isolates the information about CrewAI's specific techniques for handling memory and building agents.)

**Reasoning for this decomposition:**

*   **Focus on Core Components:** The original question deals with two fundamental aspects of both systems: memory and agents. Breaking it down allows for targeted searches for documentation detailing each.
*   **Comparative F

## Step 4: QA chain per sub-question

In [14]:
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
qa_chain = create_stuff_documents_chain(llm = llm, prompt = qa_prompt)

## Step 5: Full RAG pipeline logic

In [15]:
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]
    
    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")
    
    return "\n\n".join(results)

## Step 6: Run

In [16]:
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: Okay, here’s a decomposition of the question "How does LangChain use memory and agents compared to CrewAI?" into smaller, more manageable sub-questions for better document retrieval:
A: Okay, here’s a decomposition of the question "How does LangChain use memory and agents compared to CrewAI?" into smaller, more manageable sub-questions for better document retrieval:

*   **What is LangChain’s agent functionality?** (Focus: Understanding how LangChain agents work – LLM reasoning, tool selection, multi-step tasks, integration with tools like web search and calculators).
*   **How does LangChain handle retrieval?** (Focus: Understanding LangChain's role in retrieving information – specifically its handling of “retrieval and tool wrapping”).
*   **What is CrewAI’s role in a hybrid system with LangChain?** (Focus: Understanding CrewAI’s function – role-based collaboration).
*   **What are the key differences in the LangChain and CrewAI architectures?** (Focus: Comparing 